# 20 · Preparacion de datos — EXAMINATION

**Orden del pipeline (igual que prueba2, parando en encoding):**
```
Limpieza  ->  Imputacion (escalado + KNN)  ->  Encoding
data_cleaning   data_imputation                data_encoding
```

Se usan **los mismos nodos** de los pipelines Kedro (`src/nhanes/pipelines/...`),
alimentados con el bloque `exam` de `conf/base/parameters.yml`.

- **Input:** `data/02_intermediate/exam_intermediate.parquet` (SEQN = indice)
- **Target reservado** (no es feature): `sistolica`, `diastolica`, `presion_pulso`
- **Salida final:** `data/05_model_input/exam_encoded.parquet` (numerico, sin nulos)

> El entrenamiento (regresion / clasificacion / no supervisado) va despues.

In [1]:
import sys, yaml
from pathlib import Path
import pandas as pd

# Raiz del proyecto kedro (el notebook vive en notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

# Nodos de los 3 pipelines de preparacion
from nhanes.pipelines.data_cleaning.nodes import clean_dataset
from nhanes.pipelines.data_imputation.nodes import impute_dataset
from nhanes.pipelines.data_encoding.nodes import encode_dataset

# Parametros del dataset (mismo bloque que usa Kedro)
params = yaml.safe_load(open(ROOT / 'conf/base/parameters.yml', encoding='utf-8'))['exam']
print('Parametros exam:')
print(yaml.dump(params, allow_unicode=True, sort_keys=False))

df = pd.read_parquet(ROOT / 'data/02_intermediate/exam_intermediate.parquet')
print('Intermediate:', df.shape)
df.head()

Parametros exam:
cols_to_drop: []
target_cols:
- sistolica
- diastolica
- presion_pulso
missing_threshold: 0.6
categorical_cols_impute: []
scaler_type: standard
knn_neighbors: 5
encoding:
  binary_cols: []
  ordinal_maps: {}
  nominal_cols: []

Intermediate: (8704, 10)


,imc,cintura,altura,circ_brazo,sistolica,diastolica,pulso,presion_pulso,cintura_talla,cintura_cadera
SEQN,,,,,,,,,,
93703,17.5,48.2,NaN,16.2,NaN,NaN,NaN,NaN,NaN,NaN
93704,15.7,50.0,NaN,15.2,NaN,NaN,NaN,NaN,NaN,NaN
93705,31.7,101.8,158.3,32.0,168.5,66.0,50.0,102.5,0.643083,0.925455
93706,21.5,79.3,175.7,27.0,130.5,69.5,78.0,61.0,0.451338,0.840042
93707,18.1,64.1,158.4,21.5,136.0,71.5,90.0,64.5,0.404672,0.772289


## 1. Limpieza  (`data_cleaning`)

`clean_dataset` hace, en este orden:
1. elimina `cols_to_drop` (aqui ninguna),
2. **reserva el target** en escala original y lo separa de las features,
3. elimina features con missingness > `missing_threshold` (0.6),
4. rellena categoricas faltantes con `'Unknown'` (aqui no hay categoricas).

In [2]:
print('Nulos por columna ANTES:')
print(df.isnull().mean().round(3).sort_values(ascending=False).to_string())

exam_clean, exam_target_raw = clean_dataset(df, params)

print()
print('Features limpias:', exam_clean.shape, '->', list(exam_clean.columns))
print('Target reservado :', exam_target_raw.shape, '->', list(exam_target_raw.columns))
print()
print('Target en ESCALA ORIGINAL (no escalado):')
exam_target_raw.describe().round(1)

Nulos por columna ANTES:
pulso             0.397
cintura_cadera    0.309
sistolica         0.297
presion_pulso     0.297
diastolica        0.296
cintura_talla     0.206
altura            0.169
cintura           0.127
circ_brazo        0.093
imc               0.081

Features limpias: (8704, 7) -> ['imc', 'cintura', 'altura', 'circ_brazo', 'pulso', 'cintura_talla', 'cintura_cadera']
Target reservado : (8704, 3) -> ['sistolica', 'diastolica', 'presion_pulso']

Target en ESCALA ORIGINAL (no escalado):


,sistolica,diastolica,presion_pulso
count,6123.0,6124.0,6123.0
mean,120.3,71.6,48.7
std,19.6,12.0,14.5
min,70.0,37.5,14.0
25%,106.0,63.0,39.0
50%,117.0,70.5,45.5
75%,131.0,79.0,55.0
max,218.5,132.0,150.0


## 2. Imputacion  (`data_imputation`)

`impute_dataset` **escala** las numericas (`StandardScaler`) y luego imputa los
nulos con **KNN (k=5)**. El escalado va antes para que las distancias euclideas
del KNN sean comparables entre variables de distinto rango (misma justificacion
que prueba2).

In [3]:
print('Nulos ANTES de imputar:', int(exam_clean.isnull().sum().sum()))

exam_imputed = impute_dataset(exam_clean, params)

print('Nulos DESPUES de imputar:', int(exam_imputed.isnull().sum().sum()))
print()
print('Estadisticas post-escalado/imputacion:')
exam_imputed.describe().round(3)

Nulos ANTES de imputar: 12031


Nulos DESPUES de imputar: 0

Estadisticas post-escalado/imputacion:


,imc,cintura,altura,circ_brazo,pulso,cintura_talla,cintura_cadera
count,8704.000,8704.000,8704.000,8704.000,8704.000,8704.000,8704.000
mean,-0.039,-0.064,-0.225,-0.023,0.118,-0.156,-0.226
std,0.986,1.015,1.126,0.988,0.853,1.022,0.964
min,-1.743,-2.190,-3.022,-1.950,-3.060,-2.030,-3.254
25%,-0.789,-0.786,-0.782,-0.640,-0.413,-1.039,-1.068
50%,-0.067,-0.000,-0.004,0.021,0.095,-0.159,-0.228
75%,0.511,0.608,0.567,0.630,0.644,0.497,0.453
max,5.094,3.489,2.522,3.514,5.667,4.208,3.658


## 3. Encoding  (`data_encoding`)

`encode_dataset` aplica Label/Ordinal/One-Hot segun el bloque `encoding`.
Examination es **100% numerico**, asi que el encoding solo garantiza que no
queden columnas `bool` y deja el dataset listo para modelar.

In [4]:
exam_encoded = encode_dataset(exam_imputed, params)
print('Shape final:', exam_encoded.shape)
print('dtypes unicos:', set(exam_encoded.dtypes.astype(str)))
print('Nulos:', int(exam_encoded.isnull().sum().sum()), '| indice:', exam_encoded.index.name)
exam_encoded.head()

Shape final: (8704, 7)
dtypes unicos: {'float64'}
Nulos: 0 | indice: SEQN


,imc,cintura,altura,circ_brazo,pulso,cintura_talla,cintura_cadera
SEQN,,,,,,,
93703,-1.107026,-1.829924,-2.123088,-1.791063,-0.511602,-1.315989,-1.216772
93704,-1.327183,-1.750989,-1.165569,-1.923359,0.324219,-1.511177,-1.314712
93705,0.629765,0.520582,-0.289388,0.299221,-1.789916,0.621889,0.011349
93706,-0.617789,-0.466104,0.952105,-0.362261,0.504494,-1.132941,-0.984082
93707,-1.033641,-1.132666,-0.282253,-1.089892,1.487812,-1.560021,-1.773708


## 4. Guardado

Salidas (las mismas que produce `kedro run` via catalog):
- `03_primary/exam_clean.parquet`, `03_primary/exam_target_raw.parquet`
- `04_feature/exam_imputed.parquet`
- `05_model_input/exam_encoded.parquet`  <- entrada del modelado

In [5]:
(ROOT / 'data/03_primary').mkdir(parents=True, exist_ok=True)
(ROOT / 'data/04_feature').mkdir(parents=True, exist_ok=True)
(ROOT / 'data/05_model_input').mkdir(parents=True, exist_ok=True)

exam_clean.to_parquet(ROOT / 'data/03_primary/exam_clean.parquet')
exam_target_raw.to_parquet(ROOT / 'data/03_primary/exam_target_raw.parquet')
exam_imputed.to_parquet(ROOT / 'data/04_feature/exam_imputed.parquet')
exam_encoded.to_parquet(ROOT / 'data/05_model_input/exam_encoded.parquet')
print('Guardado OK')

Guardado OK


## Conclusiones — Examination

- De **10 columnas** del intermediate: **3 reservadas como target** (presion arterial)
  y **7 features** (antropometria + pulso).
- Sin imputacion categorica (no hay categoricas) ni One-Hot: dataset numerico puro.
- Resultado: `exam_encoded` (8704 x 7), escalado, sin nulos, `SEQN` como indice
  para unir con DEMO al final.
- **Pendiente (modelado):** construir el target de clasificacion (hipertension,
  sistolica >= 130) y de regresion (presion_pulso) desde `exam_target_raw`,
  hacer el split y entrenar.